# IBM Applied Data Science Capstone — EDA with SQL

## Objective
Run the SQL analysis against the separate IBM course SQL snapshot, `data/Spacex.csv` (101 records). This dataset is distinct from the 90-record Falcon 9 modelling dataset used in the Python EDA and machine-learning notebooks.

The SQL table contains separate `Mission_Outcome` and `Landing_Outcome` fields. Queries about booster recovery must use `Landing_Outcome`, not mission success.

In [ ]:
from pathlib import Path
import re
import sqlite3
import pandas as pd
from IPython.display import display

DATA = Path("data/Spacex.csv")
OUT = Path("outputs/sql")
OUT.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    raise FileNotFoundError(f"Place the IBM 101-row SQL exercise file at {DATA}")

df = pd.read_csv(DATA)
# Replace whitespace with underscores, but preserve the significant double underscore
# in PAYLOAD_MASS__KG_ and repair the source header "Landing _Outcome".
df.columns = [
    re.sub(r"\s+_", "_", str(c).strip()).replace(" ", "_")
    for c in df.columns
]
# IBM source dates are stored as DD-MM-YYYY; create ISO dates for reliable chronological SQL filters.
df["Launch_Date_ISO"] = pd.to_datetime(
    df["Date"], format="%d-%m-%Y", errors="coerce"
).dt.strftime("%Y-%m-%d")
if df["Launch_Date_ISO"].isna().any():
    bad_dates = df.loc[df["Launch_Date_ISO"].isna(), "Date"].drop_duplicates().tolist()
    raise ValueError(f"Unparsed source dates in Spacex.csv: {bad_dates}")

print("SQL exercise rows:", len(df))
print("Fields:", list(df.columns))
display(df.head())
assert len(df) == 101, f"Expected 101 rows in the course SQL snapshot; found {len(df)}"

## 1. Create the SPACEXTBL table and validate the load

In [ ]:
con = sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", con, if_exists="replace", index=False)
row_check = pd.read_sql_query("SELECT COUNT(*) AS Records FROM SPACEXTBL", con)
display(row_check)
assert int(row_check.loc[0, "Records"]) == 101

## 2. Unique launch sites and CCA-prefixed launch records

In [ ]:
q1 = """
SELECT DISTINCT Launch_Site
FROM SPACEXTBL
ORDER BY Launch_Site
"""
q2 = """
SELECT *
FROM SPACEXTBL
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5
"""
unique_sites = pd.read_sql_query(q1, con)
cca_examples = pd.read_sql_query(q2, con)
display(unique_sites)
display(cca_examples)
unique_sites.to_csv(OUT / "sql_unique_launch_sites.csv", index=False)

## 3. Total and average payload mass

In [ ]:
q3 = """
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_kg
FROM SPACEXTBL
WHERE Customer = 'NASA (CRS)'
"""
q4 = """
SELECT AVG(PAYLOAD_MASS__KG_) AS Average_Payload_Mass_kg
FROM SPACEXTBL
WHERE Booster_Version LIKE 'F9 v1.1%'
"""
nasa_payload = pd.read_sql_query(q3, con)
avg_f9_v11 = pd.read_sql_query(q4, con)
display(nasa_payload)
display(avg_f9_v11)

## 4. First successful ground-pad landing and drone-ship successes

In [ ]:
q5 = """
SELECT MIN(Launch_Date_ISO) AS First_Successful_Ground_Pad_Landing
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (ground pad)'
"""
q6 = """
SELECT DISTINCT Booster_Version
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000
ORDER BY Booster_Version
"""
first_ground = pd.read_sql_query(q5, con)
drone_ship = pd.read_sql_query(q6, con)
display(first_ground)
display(drone_ship)
drone_ship.to_csv(OUT / "sql_drone_ship_4000_6000.csv", index=False)

## 5. Landing outcomes and mission outcomes — separate measures

In [ ]:
q7 = """
SELECT
  CASE WHEN Landing_Outcome LIKE 'Success%' THEN 'Success'
       ELSE 'Non-success / other' END AS Landing_Outcome_Group,
  COUNT(*) AS Records
FROM SPACEXTBL
GROUP BY Landing_Outcome_Group
ORDER BY Records DESC
"""
q7_mission = """
SELECT Mission_Outcome, COUNT(*) AS Records
FROM SPACEXTBL
GROUP BY Mission_Outcome
ORDER BY Records DESC
"""
landing_counts = pd.read_sql_query(q7, con)
mission_counts = pd.read_sql_query(q7_mission, con)
display(landing_counts)
display(mission_counts)
landing_counts.to_csv(OUT / "sql_landing_outcome_summary.csv", index=False)
mission_counts.to_csv(OUT / "sql_mission_outcome_summary.csv", index=False)

## 6. Identify booster versions carrying the maximum payload

In [ ]:
q8 = """
SELECT DISTINCT Booster_Version
FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG_ = (
    SELECT MAX(PAYLOAD_MASS__KG_)
    FROM SPACEXTBL
)
ORDER BY Booster_Version
"""
max_payload_boosters = pd.read_sql_query(q8, con)
display(max_payload_boosters)
print("Maximum payload (kg):", pd.read_sql_query(
    "SELECT MAX(PAYLOAD_MASS__KG_) AS Maximum_Payload_kg FROM SPACEXTBL", con
).iloc[0,0])

## 7. Failed drone-ship landings during 2015

In [ ]:
q9 = """
SELECT Booster_Version, Launch_Site, Landing_Outcome, Date
FROM SPACEXTBL
WHERE Landing_Outcome = 'Failure (drone ship)'
  AND substr(Launch_Date_ISO, 1, 4) = '2015'
ORDER BY Date
"""
failed_2015 = pd.read_sql_query(q9, con)
display(failed_2015)
failed_2015.to_csv(OUT / "sql_failed_drone_ship_2015.csv", index=False)

## 8. Rank landing outcomes in the required date window

In [ ]:
q10 = """
SELECT Landing_Outcome, COUNT(*) AS Records
FROM SPACEXTBL
WHERE Launch_Date_ISO BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Records DESC, Landing_Outcome
"""
ranked_outcomes = pd.read_sql_query(q10, con)
display(ranked_outcomes)
ranked_outcomes.to_csv(OUT / "sql_landing_outcome_ranking_2010_2017.csv", index=False)

## 9. Final interpretation and dataset scope

The notebook executes the 101-record SQL exercise independently of the 90-record modelling dataset. It covers unique launch sites, examples filtered by site prefix, NASA payload totals, average F9 v1.1 payload, earliest successful ground-pad landing, successful drone-ship boosters in a defined payload range, landing and mission outcome counts, maximum payload booster versions, failed drone-ship attempts in 2015, and date-window ranking.

Do not interpret `Mission_Outcome` as the first-stage landing target. A successful payload mission can have a different booster recovery outcome.